In [1]:
# imports
import re
from gensim.models import Word2Vec

In [2]:
# load wikipedia data for education
def load_text_file(path):
    with open(path, "r", encoding="utf-8") as f:
        return f.read()

texts = {
    "en": load_text_file("../data/Education/English_education.txt"),
    "es": load_text_file("../data/Education/Spanish_education.txt"),
    "de": load_text_file("../data/Education/German_education.txt")
}

In [3]:
# preprocess and tokenize
def preprocess(text):
    text = text.lower()
    # text = re.sub(r'[^a-zA-Záéíóúüñßäö\s]', ' ', text)
    # text = re.sub(r'\s+', ' ', text).strip()
    return text.split()

tokenized = {
    lang: preprocess(text)
    for lang, text in texts.items()
}

In [4]:
# information about each of the wikipedia pages

stats = {}
for lang, tokens in tokenized.items():
    stats[lang] = {
        "num_words": len(tokens),
        "vocab_size": len(set(tokens))
    }

total_tokens = sum(s["num_words"] for s in stats.values())

print("Per-language stats:")
for lang, s in stats.items():
    print(lang, s)

print("Total tokens across all datasets:", total_tokens)


Per-language stats:
en {'num_words': 9169, 'vocab_size': 2383}
es {'num_words': 5316, 'vocab_size': 1831}
de {'num_words': 1074, 'vocab_size': 551}
Total tokens across all datasets: 15559


In [5]:
# train word2vec modelss
models = {}

for lang, tokens in tokenized.items():
    
    # split into pseudo-sentences (since Wikipedia text is continuous)
    sentences = [tokens[i:i+50] for i in range(0, len(tokens), 50)]
    
    model = Word2Vec(
        sentences=sentences,
        vector_size=100,
        window=5,
        min_count=2,
        workers=4
    )
    
    models[lang] = model

In [6]:
print("English closest words to 'education':")
print(models["en"].wv.most_similar("education", topn=5))

print("Spanish closest words to 'education':")
print(models["es"].wv.most_similar("educación", topn=5))

print("German closest words to 'education':")
print(models["de"].wv.most_similar("ausbildung", topn=5))

English closest words to 'education':
[('of', 0.9996594786643982), ('and', 0.9996375441551208), ('the', 0.999626874923706), ('to', 0.9995654225349426), ('by', 0.9994922280311584)]
Spanish closest words to 'education':
[('y', 0.9949256181716919), ('de', 0.9945831298828125), ('en', 0.9943817257881165), ('la', 0.9941930770874023), ('a', 0.9939391613006592)]
German closest words to 'education':
[('stunden', 0.37181466817855835), ('durch', 0.3089267909526825), ('interesse', 0.26361313462257385), ('verlängerung', 0.24354466795921326), ('werden.', 0.19456851482391357)]
